# Online Retail: from raw transactions to business decisions

Independent portfolio analysis using Chen (2015), UCI Online Retail, CC BY 4.0. Currency: GBP. See the case study for source citation, assumptions and limitations.

**Objective:** quantify sales, credit exposure and repeat purchasing, then propose measurable business actions.

In [1]:
from pathlib import Path
import json, sqlite3
import pandas as pd
ROOT = Path.cwd() if (Path.cwd() / 'reports/metrics.json').exists() else Path.cwd().parent
k = json.loads((ROOT / 'reports/metrics.json').read_text())
audit = json.loads((ROOT / 'reports/data_quality.json').read_text())
print('Observation window:', k['start'], 'to', k['end'])
print('Source SHA-256:', audit['source_sha256'])

Observation window: 2010-12-01 to 2011-12-09
Source SHA-256: 43465a06f2ccf7c8b5bd2892bc7defb52f97487934fe93b16ae4c3936424676d


## 1. Audit the cleaning decisions

The pipeline removes exact duplicates as an assumption, excludes nonpositive prices and invalid transactional records, and retains negative quantities as credits. Anonymous purchases remain in sales totals. No outliers are trimmed. The source workbook is not modified.

In [2]:
print(json.dumps(audit, indent=2))
assert audit['raw_rows'] == audit['duplicate_rows_removed'] + audit['excluded_rows_after_dedup'] + audit['included_rows']
assert abs(audit['raw_signed_value'] - audit['duplicate_signed_value'] - audit['excluded_signed_value'] - k['net_sales']) < .01
print('Row and monetary reconciliation passed.')

{
  "raw_rows": 541909,
  "duplicate_rows_removed": 5268,
  "excluded_rows_after_dedup": 2512,
  "included_rows": 534129,
  "missing_customer_rows_raw": 135080,
  "raw_signed_value": 9747747.934,
  "duplicate_signed_value": 21740.98,
  "excluded_signed_value": -22124.12,
  "exclusions": {
    "Nonpositive or missing price": 2512
  },
  "source_sha256": "43465a06f2ccf7c8b5bd2892bc7defb52f97487934fe93b16ae4c3936424676d",
  "net_sales_without_dedup": 9769872.054,
  "validation": "Passed: row accounting, monetary bridge, gross less credits, invoice grain, SQL/Python KPI agreement"
}
Row and monetary reconciliation passed.


## 2. Reconcile the headline metrics in SQL

Net sales is signed transaction value, not profit. Gross AOV divides positive sales by distinct purchase invoices. A credit-value ratio does not establish a matched product-return rate.

In [3]:
con = sqlite3.connect(ROOT / 'data/retail.sqlite')
query = '''SELECT SUM(GrossSales) AS GrossSales, SUM(Credits) AS Credits, SUM(NetSales) AS NetSales, COUNT(DISTINCT CASE WHEN IsSale=1 THEN InvoiceNo END) AS Orders FROM transactions'''
headline = pd.read_sql_query(query, con)
print(headline.round(2).to_string(index=False))
assert abs(headline.NetSales.iloc[0] - k['net_sales']) < .01
print('SQL agrees with Python.')

 GrossSales   Credits   NetSales  Orders
 10642110.8 893979.73 9748131.07   19960
SQL agrees with Python.


## 3. Compare complete months

The last month ends on 9 December 2011 and is excluded from the growth comparison. One year of activity is insufficient to establish a recurring seasonal effect.

In [4]:
monthly = pd.read_csv(ROOT / 'data/processed/monthly_sales.csv')
complete = monthly[monthly.Month < k['end'][:7]].copy()
complete['MoMPercent'] = complete.NetSales.pct_change() * 100
print(complete[['Month','NetSales','MoMPercent']].round(2).to_string(index=False))

  Month   NetSales  MoMPercent
2010-12  746723.61         NaN
2011-01  558448.56      -25.21
2011-02  497026.41      -11.00
2011-03  682013.98       37.22
2011-04  492367.84      -27.81
2011-05  722094.10       46.66
2011-06  689977.23       -4.45
2011-07  680156.99       -1.42
2011-08  703510.58        3.43
2011-09 1017596.68       44.65
2011-10 1069368.23        5.09
2011-11 1456145.80       36.17


## 4. Identify market and product concentration

Overall sales include postage and special codes. Product rankings apply the five-digit stock-code heuristic, so the two totals have different scopes.

In [5]:
markets = pd.read_csv(ROOT / 'data/processed/country_sales.csv')
products = pd.read_csv(ROOT / 'data/processed/products.csv')
print(markets.head(8).round(2).to_string(index=False))
print(products[['StockCode','Description','NetSales']].head(8).round(2).to_string(index=False))

       Country  GrossSales   Credits   NetSales  Orders
United Kingdom  9001744.09 812491.79 8189252.30   18019
   Netherlands   285446.34    784.80  284661.54      94
          EIRE   283140.52  20147.14  262993.38     288
       Germany   228678.40   7168.93  221509.47     457
        France   209625.37  12308.26  197317.11     392
     Australia   138453.81   1444.04  137009.77      57
   Switzerland    57067.60    704.55   56363.05      54
         Spain    61558.56   6802.53   54756.03      90
StockCode                        Description  NetSales
    22423           REGENCY CAKESTAND 3 TIER 164459.49
    47566                      PARTY BUNTING  98243.88
   85123A WHITE HANGING HEART T-LIGHT HOLDER  97838.45
   85099B            JUMBO BAG RED RETROSPOT  92175.79
    23084                 RABBIT NIGHT LIGHT  66661.63
    22086     PAPER CHAIN KIT 50'S CHRISTMAS  63715.24
    84879      ASSORTED COLOUR BIRD ORNAMENT  58792.42
    79321                      CHILLI LIGHTS  53746.66


## 5. Understand repeat purchasing and target groups

Customer analysis excludes anonymous transactions. Frequency counts distinct purchase invoices, monetary value is gross spend, and recency is measured at 10 December 2011. Fixed thresholds create explainable groups; they are not a predictive churn model.

In [6]:
rfm = pd.read_csv(ROOT / 'data/processed/customer_segments.csv')
summary = rfm.groupby('Segment').agg(Customers=('CustomerID','size'), GrossSales=('Monetary','sum'))
print(summary.round(2).to_string())
print('Repeat buyer share:', f"{(rfm.Frequency >= 2).mean():.1%}")
print('Top 10 share of identified gross sales:', f"{rfm.nlargest(10, 'Monetary').Monetary.sum()/rfm.Monetary.sum():.1%}")

                 Customers  GrossSales
Segment                               
Active frequent        787  5338740.58
Active repeat         1456  2258176.26
Lapsed repeat          602   676302.49
Other customers       1286   549574.46
Recent one-time        207    64415.10
Repeat buyer share: 65.6%
Top 10 share of identified gross sales: 17.3%


## 6. Distinguish zero retention from unobserved time

Cohorts start at first observed purchase. Missing cells are future or partial periods, not zero. December 2011 is excluded from retention follow-up. Existing customers at the beginning of the dataset may appear in the first cohort.

In [7]:
cohort = pd.read_csv(ROOT / 'data/processed/cohort_retention.csv')
heatmap = cohort.pivot(index='Cohort',columns='MonthIndex',values='Retention')
print((heatmap*100).round(1).to_string(na_rep='not observed'))
assert cohort.Retention.dropna().between(0,1).all()
con.close()

MonthIndex           0            1            2            3            4            5            6            7            8            9            10           11           12
Cohort                                                                                                                                                                             
2010-12           100.0         36.6         32.3         38.4         36.3         39.8         36.3         34.9         35.4         39.5         37.4         50.3 not observed
2011-01           100.0         22.1         26.6         23.0         32.1         28.8         24.7         24.2         30.0         32.6         36.5 not observed not observed
2011-02           100.0         18.7         18.7         28.4         27.1         24.7         25.3         27.9         24.7         30.5 not observed not observed not observed
2011-03           100.0         15.0         25.2         19.9         22.3         16.8         26.

## 7. Business recommendations

- Plan inventory around the observed peak period, then validate against stockout and fulfilment data.
- Test reactivation of lapsed repeat customers using a holdout group.
- Review large credit invoices and collect a reason code before changing return policies.
- Improve CustomerID capture to increase the coverage of customer analysis.

No uplift has been measured. Cost, web traffic and advertising data are absent, so profit, conversion and ROAS cannot be inferred. Open `dashboard/index.html` for the interactive views and read `reports/CASE_STUDY.md` for numerical findings.